# [STARTER] Udaplay Project

## Part 02 - Agent

In this part of the project, you'll use your VectorDB to be part of your Agent as a tool.

You're building UdaPlay, an AI Research Agent for the video game industry. The agent will:
1. Answer questions using internal knowledge (RAG)
2. Search the web when needed
3. Maintain conversation state
4. Return structured outputs
5. Store useful information for future use

### Setup

In [19]:
import os
import chromadb
from enum import Enum
from typing import Optional
from pydantic import BaseModel, Field
from tavily import TavilyClient
from dotenv import load_dotenv
from chromadb.utils import embedding_functions

In [4]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

In [7]:
CHROMA_PATH = "chroma_db"
COLLECTION_NAME = "games_collection"

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME, embedding_function=embedding_fn)

### Tools

Build at least 3 tools:
- retrieve_game: To search the vector DB
- evaluate_retrieval: To assess the retrieval performance
- game_web_search: If no good, search the web


#### Retrieve Game Tool

In [8]:
def retrieve_game(query: str, n_results: int = 3) -> list[dict]:
    """
    Semantic search: Finds the most relevant results in the vector DB.

    Args:
        query: A question about the game industry.
        n_results: Number of results to retrieve.

    Returns:
        A list of dictionaries. Each dictionary contains:
        - Platform: platform of the game
        - Name: name of the game
        - YearOfRelease: release year
        - Description: additional details
        - document: indexed text used in the vector database
        - distance: semantic distance from the query
    """

    results = collection.query(
        query_texts=[query],
        n_results=n_results
    )

    retrieved_games = []

    documents = results["documents"][0]
    metadatas = results["metadatas"][0]
    distances = results["distances"][0]

    for document, metadata, distance in zip(documents, metadatas, distances):
        retrieved_games.append({
            "Name": metadata.get("Name"),
            "Platform": metadata.get("Platform"),
            "YearOfRelease": metadata.get("YearOfRelease"),
            "Description": metadata.get("Description"),
            "document": document,
            "distance": distance
        })

    return retrieved_games

In [9]:
results = retrieve_game("Which game was released for Nintendo 64?")

for game in results:
    print(f"Name: {game['Name']}")
    print(f"Platform: {game['Platform']}")
    print(f"Year: {game['YearOfRelease']}")
    print(f"Description: {game['Description']}")
    print(f"Distance: {game['distance']}")
    print("-" * 50)

Name: Super Mario 64
Platform: Nintendo 64
Year: 1996
Description: A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.
Distance: 0.3255818486213684
--------------------------------------------------
Name: Super Mario World
Platform: Super Nintendo Entertainment System (SNES)
Year: 1990
Description: A classic platformer where Mario embarks on a quest to save Princess Toadstool and Dinosaur Land from Bowser.
Distance: 0.5084282159805298
--------------------------------------------------
Name: Super Smash Bros. Melee
Platform: GameCube
Year: 2001
Description: A crossover fighting game featuring characters from various Nintendo franchises battling it out in dynamic arenas.
Distance: 0.5771632194519043
--------------------------------------------------


### Evaluate Retrieval Tool

This tool evaluates whether the retrieved documents are useful enough to answer the user's question.

The project instructions mention that an LLM can be used as a judge. However, to keep the tool reliable and avoid API quota issues during development, this implementation uses the semantic distance returned by ChromaDB.

A lower distance means the retrieved document is more semantically similar to the question. If the best retrieved document has a distance below the defined threshold, the retrieval is considered useful. Otherwise, the agent should fall back to web search.

In [ ]:
class EvaluationReport(BaseModel):
    useful: bool = Field(description="Whether the retrieved documents are useful enough to answer the user's question.")

    description: str = Field(description="Detailed explanation of why the documents are or are not useful.")

In [ ]:
def evaluate_retrieval(question: str, retrieved_docs: list[dict], max_distance: float = 0.45) -> EvaluationReport:

    if not retrieved_docs:
        return EvaluationReport(
            useful=False,
            description=(
                "No documents were retrieved from the vector database. "
                "The retrieved list is empty, so the agent should use web search "
                "or another external source to answer the question."
            )
        )

    best_doc = retrieved_docs[0]
    best_distance = best_doc.get("distance")

    if best_distance is None:
        return EvaluationReport(
            useful=False,
            description=(
                "The retrieved documents do not include a semantic distance score. "
                "Without this score, the tool cannot safely evaluate retrieval quality. "
                "The agent should use web search or another external source."
            )
        )

    if best_distance <= max_distance:
        return EvaluationReport(
            useful=True,
            description=(
                f"The retrieved documents are useful for answering the question. "
                f"The best result is '{best_doc.get('Name')}' on platform "
                f"'{best_doc.get('Platform')}', released in "
                f"{best_doc.get('YearOfRelease')}. Its semantic distance is "
                f"{best_distance:.4f}, which is below the threshold of "
                f"{max_distance}. This means the result is relevant enough to answer "
                f"the question: '{question}'."
            )
        )

    return EvaluationReport(
        useful=False,
        description=(
            f"The retrieved documents may not be useful enough to answer the question. "
            f"The best result is '{best_doc.get('Name')}' on platform "
            f"'{best_doc.get('Platform')}', but its semantic distance is "
            f"{best_distance:.4f}, which is above the threshold of {max_distance}. "
            f"The agent should use web search to find a more reliable answer."
        )
    )

In [14]:
question = "Which game was released for Nintendo 64?"

retrieved_docs = retrieve_game(question)

evaluation = evaluate_retrieval(question, retrieved_docs)

print("Useful:", evaluation.useful)
print("Description:", evaluation.description)

Useful: True
Description: The retrieved documents are useful for answering the question. The best result is 'Super Mario 64' on platform 'Nintendo 64', released in 1996. Its semantic distance is 0.3256, which is below the threshold of 0.45. This means the result is relevant enough to answer the question: 'Which game was released for Nintendo 64?'.


#### Game Web Search Tool

In [15]:
tavily_client = TavilyClient(api_key=TAVILY_API_KEY)


def game_web_search(question: str, max_results: int = 3) -> list[dict]:

    search_query = f"video game industry {question}"

    response = tavily_client.search(query=search_query, search_depth="basic",  max_results=max_results)

    web_results = []

    for result in response.get("results", []):
        web_results.append({
            "title": result.get("title"),
            "url": result.get("url"),
            "content": result.get("content")
        })

    return web_results

In [16]:
question = "Was Mortal Kombat X released for PlayStation 5?"

web_results = game_web_search(question)

for result in web_results:
    print("Title:", result["title"])
    print("URL:", result["url"])
    print("Content:", result["content"])
    print("-" * 50)

Title: Mortal Kombat X - Wikipedia
URL: https://en.wikipedia.org/wiki/Mortal_Kombat_X
Content: ***Mortal Kombat X*** is a 2015 fighting game developed by NetherRealm Studios and published by Warner Bros. An upgraded version of *Mortal Kombat X*, titled ***Mortal Kombat XL***, was released on March 1, 2016, for PlayStation 4 and Xbox One, including all downloadable content characters from the two released Kombat Packs, almost all bonus alternate costumes available at the time of release, improved gameplay, and improved netcode. By July 2015, due to heavy criticism for the porting issues that plagued the PC release of the game, almost all references to *Mortal Kombat X* had been removed from High Voltage Software's Facebook page. On March 2, 2015, NetherRealm Studios announced that their mobile division would release an iOS/Android "Android (operating system)") version of *Mortal Kombat X* in April 2015. With the 1.11 update version of the mobile game released on December 6, 2016, Freddy

### Agent

In [ ]:
class AgentState(str, Enum):
    START = "start"
    RETRIEVE = "retrieve"
    EVALUATE = "evaluate"
    WEB_SEARCH = "web_search"
    RESPOND = "respond"

class AgentResponse(BaseModel):

    question: str = Field(description="Original user question.")
    answer: str = Field(description="Final answer to the user.")
    source: str = Field(description="Source used to answer: vector_database or web_search.")
    tool_trace: list[str] = Field(description="List of tools used by the agent.")
    citation: Optional[str] = Field(default=None, description="Citation or source URL, if available.")
    retrieved_docs: Optional[list[dict]] = Field(default=None, description="Documents retrieved from the vector database.")
    web_results: Optional[list[dict]] = Field(default=None, description="Results retrieved from web search.")


In [ ]:
class UdaPlayAgent:

    def __init__(self):
        self.state = AgentState.START
        self.conversation_history = []

    def answer_from_retrieval(self, question: str, retrieved_docs: list[dict]) -> str:

        best_doc = retrieved_docs[0]

        name = best_doc.get("Name")
        platform = best_doc.get("Platform")
        year = best_doc.get("YearOfRelease")
        description = best_doc.get("Description")

        return (
            f"Based on the local game database, the most relevant result is "
            f"{name}, released for {platform} in {year}. "
            f"{description}"
        )

    def answer_from_web(self, question: str, web_results: list[dict]) -> tuple[str, Optional[str]]:

        if not web_results:
            return (
                "I could not find enough information in the local database or through web search "
                "to answer this question confidently.",
                None
            )

        best_result = web_results[0]

        title = best_result.get("title")
        content = best_result.get("content")
        url = best_result.get("url")

        answer = (
            f"Based on web search, the most relevant source I found is '{title}'. "
            f"{content}"
        )

        return answer, url

    def run(self, question: str) -> AgentResponse:

        tool_trace = []

        self.state = AgentState.START
        tool_trace.append("Agent started.")

        self.state = AgentState.RETRIEVE
        tool_trace.append("Using retrieve_game to search the local vector database.")
        retrieved_docs = retrieve_game(question)

        self.state = AgentState.EVALUATE
        tool_trace.append("Using evaluate_retrieval to check if retrieved documents are useful.")
        evaluation = evaluate_retrieval(question, retrieved_docs)

        if evaluation.useful:
            self.state = AgentState.RESPOND
            tool_trace.append("Retrieved documents were considered useful.")
            tool_trace.append("Answering using the local vector database.")

            answer = self.answer_from_retrieval(question, retrieved_docs)

            response = AgentResponse(
                question=question,
                answer=answer,
                source="vector_database",
                tool_trace=tool_trace,
                citation=retrieved_docs[0].get("document"),
                retrieved_docs=retrieved_docs,
                web_results=None
            )

        else:
            self.state = AgentState.WEB_SEARCH
            tool_trace.append("Retrieved documents were not useful enough.")
            tool_trace.append("Using game_web_search to search external sources.")

            web_results = game_web_search(question)

            self.state = AgentState.RESPOND
            tool_trace.append("Answering using web search results.")

            answer, citation = self.answer_from_web(question, web_results)

            response = AgentResponse(
                question=question,
                answer=answer,
                source="web_search",
                tool_trace=tool_trace,
                citation=citation,
                retrieved_docs=retrieved_docs,
                web_results=web_results
            )

        self.conversation_history.append({
            "question": question,
            "answer": response.answer,
            "source": response.source
        })

        return response

    def show_history(self):
        return self.conversation_history

In [22]:
agent = UdaPlayAgent()

In [23]:
response = agent.run("When Pokémon Gold and Silver was released?")

print("Question:", response.question)
print("Answer:", response.answer)
print("Source:", response.source)
print("Citation:", response.citation)
print("\nTool Trace:")
for step in response.tool_trace:
    print("-", step)

Question: When Pokémon Gold and Silver was released?
Answer: Based on the local game database, the most relevant result is Pokémon Gold and Silver, released for Game Boy Color in 1999. Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.
Source: vector_database
Citation: [Game Boy Color] Pokémon Gold and Silver (1999) - Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.

Tool Trace:
- Agent started.
- Using retrieve_game to search the local vector database.
- Using evaluate_retrieval to check if retrieved documents are useful.
- Retrieved documents were considered useful.
- Answering using the local vector database.


In [24]:
questions = [
    "When Pokémon Gold and Silver was released?",
    "Which one was the first 3D platformer Mario game?",
    "Was Mortal Kombat X released for PlayStation 5?"
]

for question in questions:
    response = agent.run(question)

    print("=" * 80)
    print("Question:", response.question)
    print("Answer:", response.answer)
    print("Source:", response.source)
    print("Citation:", response.citation)

    print("\nTool Trace:")
    for step in response.tool_trace:
        print("-", step)

    print()

Question: When Pokémon Gold and Silver was released?
Answer: Based on the local game database, the most relevant result is Pokémon Gold and Silver, released for Game Boy Color in 1999. Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.
Source: vector_database
Citation: [Game Boy Color] Pokémon Gold and Silver (1999) - Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.

Tool Trace:
- Agent started.
- Using retrieve_game to search the local vector database.
- Using evaluate_retrieval to check if retrieved documents are useful.
- Retrieved documents were considered useful.
- Answering using the local vector database.

Question: Which one was the first 3D platformer Mario game?
Answer: Based on the local game database, the most relevant result is Super Mario 64, released for Nintendo 64 in 1996. A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.


In [25]:
agent.show_history()

[{'question': 'When Pokémon Gold and Silver was released?',
  'answer': 'Based on the local game database, the most relevant result is Pokémon Gold and Silver, released for Game Boy Color in 1999. Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.',
  'source': 'vector_database'},
 {'question': 'When Pokémon Gold and Silver was released?',
  'answer': 'Based on the local game database, the most relevant result is Pokémon Gold and Silver, released for Game Boy Color in 1999. Second-generation Pokémon games introducing new regions, Pokémon, and gameplay mechanics.',
  'source': 'vector_database'},
 {'question': 'Which one was the first 3D platformer Mario game?',
  'answer': "Based on the local game database, the most relevant result is Super Mario 64, released for Nintendo 64 in 1996. A groundbreaking 3D platformer that set new standards for the genre, featuring Mario's quest to rescue Princess Peach.",
  'source': 'vector_database'},
 {'question':

### (Optional) Advanced

In [ ]:
# TODO: Update your agent with long-term memory
# TODO: Convert the agent to be a state machine, with the tools being pre-defined nodes